# Consumer-360: Portable BI Standardization Pipeline (Row Values, Table Headings & Supabase Export)

This portable notebook standardizes all five Consumer-360 CSVs for **Supabase** and **Streamlit** BI presentation and exports them directly to `<PROJECT_ROOT>/data/supabase_db/` with clean BI filenames:

| Source CSV | Supabase BI CSV | Transformations Applied |
| :--- | :--- | :--- |
| `cleaned_retail_week1.csv` | `retail_transactions.csv` | Standardizes `Country` & `Product Name` values; renames all 11 table headings |
| `customer_rfm_week1.csv` | `customer_rfm_metrics.csv` | Standardizes `Country` values; renames all 12 table headings |
| `customer_segments_week2.csv` | `customer_segments.csv` | Standardizes `Country`, `Customer Tier`, & `Buyer Group`; renames all 14 table headings |
| `product_affinity_week2.csv` | `product_bundles.csv` | Standardizes `Product A` & `Product B` names; renames all 11 table headings |
| `cohort_retention_week2.csv` | `cohort_retention.csv` | Keeps `Months Since First Purchase` numeric (`0–12`); renames all 5 table headings |

> **Portability:** Works seamlessly whether run from `notebooks/`, the project root, or any working directory.

In [1]:
import re
from pathlib import Path

import pandas as pd

# Fallback display() for non-IPython / headless environments
try:
    display  # type: ignore[name-defined]
except NameError:
    def display(obj):  # type: ignore[misc]
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

# Resolve project root cleanly (supports running from notebooks/ or repo root)
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD

# Auto-detect input directory containing the processed CSVs
CANDIDATE_DIRS = [
    PROJECT_ROOT / "data" / "processed",
    CWD / "data" / "processed",
    Path("../data/processed"),
    Path("user_uploaded_files/ttl=63d/user_uploaded"),
    CWD,
]
INPUT_DIR = next((p for p in CANDIDATE_DIRS if (p / "product_affinity_week2.csv").exists()), CWD)

# Export directly to <PROJECT_ROOT>/data/supabase_db
SUPABASE_DIR = PROJECT_ROOT / "data" / "supabase_db"
SUPABASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root    : {PROJECT_ROOT.resolve()}")
print(f"Input directory : {INPUT_DIR.resolve()}")
print(f"Supabase output : {SUPABASE_DIR.resolve()}")

Project root    : C:\Users\acer\Downloads\Consumer-360
Input directory : C:\Users\acer\Downloads\Consumer-360\data\processed
Supabase output : C:\Users\acer\Downloads\Consumer-360\data\supabase_db


In [ ]:
# 1. Output CSV filename mapping (technical -> BI Supabase table filenames)
CSV_FILE_RENAME_MAP = {
    "cleaned_retail_week1.csv": "retail_transactions.csv",
    "customer_rfm_week1.csv": "customer_rfm_metrics.csv",
    "customer_segments_week2.csv": "customer_segments.csv",
    "product_affinity_week2.csv": "product_bundles.csv",
    "cohort_retention_week2.csv": "cohort_retention.csv",
}

# 2. Country code / dataset term -> BI common name
COUNTRY_BI_MAP = {
    "EIRE": "Ireland",
    "RSA": "South Africa",
    "USA": "United States",
    "Korea": "South Korea",
    "Unspecified": "Not Specified",
}

# 3. K-Means numeric cluster ID -> BI buyer persona (based on cluster RFM profiles)
# Cluster 3: avg recency 13.8d, freq 13.5, spend 7,310
# Cluster 1: avg recency 82.2d, freq 4.1,  spend 1,787
# Cluster 0: avg recency 22.4d, freq 2.1,  spend 560
# Cluster 2: avg recency 184.7d, freq 1.3, spend 302
KMEANS_CLUSTER_BI_MAP = {
    3: "VIP High-Value Buyers",
    "3": "VIP High-Value Buyers",
    1: "Established Regulars",
    "1": "Established Regulars",
    0: "Promising Recent Buyers",
    "0": "Promising Recent Buyers",
    2: "Dormant Low-Activity",
    "2": "Dormant Low-Activity",
}

# 4. RFM segment -> BI-friendly label
RFM_SEGMENT_BI_MAP = {
    "Hibernating / Lost": "Inactive / Lost",
    "Potential Loyalists": "Potential Loyalists",
    "Loyal Customers": "Loyal Customers",
    "At Risk": "At Risk",
    "Champions": "Champions",
}

# 5. Technical column headings -> BI common table headings (per dataset)
RETAIL_COL_MAP = {
    "invoice": "Invoice No",
    "stock_code": "Product Code",
    "description": "Product Name",
    "quantity": "Quantity",
    "price": "Unit Price",
    "customer_id": "Customer ID",
    "country": "Country",
    "invoice_date": "Order Date",
    "line_revenue": "Line Revenue",
    "year_month": "Year-Month",
    "hour": "Order Hour",
}

RFM_COL_MAP = {
    "customer_id": "Customer ID",
    "last_purchase": "Last Purchase",
    "frequency": "Total Orders",
    "monetary": "Total Spend",
    "total_units": "Total Units",
    "country": "Country",
    "recency": "Days Inactive",
    "avg_order_value": "Avg Order Value",
    "r_score": "Recency Score",
    "f_score": "Frequency Score",
    "m_score": "Spend Score",
    "rfm_score": "Overall Score",
}

SEGMENTS_COL_MAP = {
    "customer_id": "Customer ID",
    "country": "Country",
    "last_purchase": "Last Purchase",
    "recency": "Days Inactive",
    "frequency": "Total Orders",
    "monetary": "Total Spend",
    "total_units": "Total Units",
    "avg_order_value": "Avg Order Value",
    "r_score": "Recency Score",
    "f_score": "Frequency Score",
    "m_score": "Spend Score",
    "rfm_score": "Overall Score",
    "rfm_segment": "Customer Tier",
    "kmeans_cluster": "Buyer Group",
}

AFFINITY_COL_MAP = {
    "stock_code_a": "Product A Code",
    "description_a": "Product A",
    "stock_code_b": "Product B Code",
    "description_b": "Product B",
    "pair_count": "Orders Together",
    "item_a_count": "Product A Orders",
    "item_b_count": "Product B Orders",
    "support_pct": "Order Share (%)",
    "confidence_a_to_b_pct": "Buy A -> Buy B (%)",
    "confidence_b_to_a_pct": "Buy B -> Buy A (%)",
    "lift": "Bundle Lift",
}

COHORT_COL_MAP = {
    "cohort_month": "First Purchase Month",
    "cohort_index": "Months Since First Purchase",
    "active_customers": "Active Customers",
    "cohort_size": "Cohort Size",
    "retention_pct": "Retention Rate (%)",
}

# Token replacements applied after Title Case for product descriptions
PRODUCT_TOKEN_FIXES = {
    "T-light": "T-Light",
    "T-lights": "T-Lights",
    "Led": "LED",
    "Usb": "USB",
    "Pvc": "PVC",
    "Dvd": "DVD",
    "Bbq": "BBQ",
    "Uk": "UK",
    "Usa": "USA",
    "3d": "3D",
    "2d": "2D",
    "Of": "of",
    "And": "and",
    "With": "with",
    "In": "in",
    "On": "on",
    "For": "for",
    "To": "to",
    "The": "the",
    "A": "a",
}


def clean_product_name(val: object) -> object:
    if pd.isna(val):
        return val
    text = str(val).strip()
    if not text:
        return text
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    text = re.sub(r"\s*/\s*", "/", text)
    text = text.rstrip("., ")
    words = text.lower().title().split(" ")
    cleaned_words = []
    for idx, word in enumerate(words):
        word = re.sub(r"^(\d+)(Cm|Mm|Ml|G|Kg|L)$", lambda m: m.group(1) + m.group(2).lower(), word)
        if idx > 0 and word in PRODUCT_TOKEN_FIXES:  # noqa: SIM114
            word = PRODUCT_TOKEN_FIXES[word]
        elif idx == 0 and word in {"Led", "Usb", "Pvc", "Dvd", "Bbq", "Uk", "Usa", "3d", "2d", "T-light", "T-lights"}:
            word = PRODUCT_TOKEN_FIXES[word]
        cleaned_words.append(word)
    return " ".join(cleaned_words)

In [3]:
# Process 1/5: cleaned_retail_week1.csv -> retail_transactions.csv
retail_df = pd.read_csv(INPUT_DIR / "cleaned_retail_week1.csv")
if "country" in retail_df.columns:
    retail_df["country"] = retail_df["country"].replace(COUNTRY_BI_MAP)
if "description" in retail_df.columns:
    retail_df["description"] = retail_df["description"].map(clean_product_name)
retail_df = retail_df.rename(columns=RETAIL_COL_MAP)

out_path = SUPABASE_DIR / CSV_FILE_RENAME_MAP["cleaned_retail_week1.csv"]
retail_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved {out_path.name}: {retail_df.shape}")
print("Headings:", list(retail_df.columns))
display(retail_df.head(5))

Saved retail_transactions.csv: (399568, 11)
Headings: ['Invoice No', 'Product Code', 'Product Name', 'Quantity', 'Unit Price', 'Customer ID', 'Country', 'Order Date', 'Line Revenue', 'Year-Month', 'Order Hour']


,Invoice No,Product Code,Product Name,Quantity,Unit Price,Customer ID,Country,Order Date,Line Revenue,Year-Month,Order Hour
0,489434,85048,15cm Christmas Glass Ball 20 Lights,12,6.95,13085,United Kingdom,2009-12-01 07:45:00,83.4,2009-12,7
1,489434,79323P,Pink Cherry Lights,12,6.75,13085,United Kingdom,2009-12-01 07:45:00,81.0,2009-12,7
2,489434,79323W,White Cherry Lights,12,6.75,13085,United Kingdom,2009-12-01 07:45:00,81.0,2009-12,7
3,489434,22041,"Record Frame 7"" Single Size",48,2.10,13085,United Kingdom,2009-12-01 07:45:00,100.8,2009-12,7
4,489434,21232,Strawberry Ceramic Trinket Box,24,1.25,13085,United Kingdom,2009-12-01 07:45:00,30.0,2009-12,7


In [4]:
# Process 2/5: customer_rfm_week1.csv -> customer_rfm_metrics.csv
rfm_df = pd.read_csv(INPUT_DIR / "customer_rfm_week1.csv")
if "country" in rfm_df.columns:
    rfm_df["country"] = rfm_df["country"].replace(COUNTRY_BI_MAP)
rfm_df = rfm_df.rename(columns=RFM_COL_MAP)

out_path = SUPABASE_DIR / CSV_FILE_RENAME_MAP["customer_rfm_week1.csv"]
rfm_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved {out_path.name}: {rfm_df.shape}")
print("Headings:", list(rfm_df.columns))
display(rfm_df.head(5))

Saved customer_rfm_metrics.csv: (4285, 12)
Headings: ['Customer ID', 'Last Purchase', 'Total Orders', 'Total Spend', 'Total Units', 'Country', 'Days Inactive', 'Avg Order Value', 'Recency Score', 'Frequency Score', 'Spend Score', 'Overall Score']


,Customer ID,Last Purchase,Total Orders,Total Spend,Total Units,Country,Days Inactive,Avg Order Value,Recency Score,Frequency Score,Spend Score,Overall Score
0,12346,2010-06-28 13:53:00,2,169.36,24,United Kingdom,165,84.68,1,2,1,4
1,12347,2010-12-07 14:57:00,2,1323.32,828,Iceland,3,661.66,4,2,3,9
2,12348,2010-09-27 14:59:00,1,221.16,372,Finland,74,221.16,2,1,1,4
3,12349,2010-10-28 08:23:00,2,2221.14,991,Italy,43,1110.57,3,2,4,9
4,12351,2010-11-29 15:23:00,1,300.93,261,Not Specified,11,300.93,4,1,1,6


In [5]:
# Process 3/5: customer_segments_week2.csv -> customer_segments.csv
seg_df = pd.read_csv(INPUT_DIR / "customer_segments_week2.csv")
if "country" in seg_df.columns:
    seg_df["country"] = seg_df["country"].replace(COUNTRY_BI_MAP)
if "rfm_segment" in seg_df.columns:
    seg_df["rfm_segment"] = seg_df["rfm_segment"].replace(RFM_SEGMENT_BI_MAP)
if "kmeans_cluster" in seg_df.columns:
    seg_df["kmeans_cluster"] = seg_df["kmeans_cluster"].replace(KMEANS_CLUSTER_BI_MAP)
seg_df = seg_df.rename(columns=SEGMENTS_COL_MAP)

out_path = SUPABASE_DIR / CSV_FILE_RENAME_MAP["customer_segments_week2.csv"]
seg_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved {out_path.name}: {seg_df.shape}")
print("Headings:", list(seg_df.columns))
print("\nBuyer Group Distribution:")
print(seg_df["Buyer Group"].value_counts())
display(seg_df.head(5))

Saved customer_segments.csv: (4285, 14)
Headings: ['Customer ID', 'Country', 'Last Purchase', 'Days Inactive', 'Total Orders', 'Total Spend', 'Total Units', 'Avg Order Value', 'Recency Score', 'Frequency Score', 'Spend Score', 'Overall Score', 'Customer Tier', 'Buyer Group']

Buyer Group Distribution:
Buyer Group
Dormant Low-Activity       1408
Established Regulars       1174
Promising Recent Buyers     938
VIP High-Value Buyers       765
Name: count, dtype: int64


,Customer ID,Country,Last Purchase,Days Inactive,Total Orders,Total Spend,Total Units,Avg Order Value,Recency Score,Frequency Score,Spend Score,Overall Score,Customer Tier,Buyer Group
0,12346,United Kingdom,2010-06-28 13:53:00,165,2,169.36,24,84.68,1,2,1,4,Inactive / Lost,Dormant Low-Activity
1,12347,Iceland,2010-12-07 14:57:00,3,2,1323.32,828,661.66,4,2,3,9,Potential Loyalists,Promising Recent Buyers
2,12348,Finland,2010-09-27 14:59:00,74,1,221.16,372,221.16,2,1,1,4,Inactive / Lost,Dormant Low-Activity
3,12349,Italy,2010-10-28 08:23:00,43,2,2221.14,991,1110.57,3,2,4,9,Potential Loyalists,Established Regulars
4,12351,Not Specified,2010-11-29 15:23:00,11,1,300.93,261,300.93,4,1,1,6,Potential Loyalists,Promising Recent Buyers


In [6]:
# Process 4/5: product_affinity_week2.csv -> product_bundles.csv
affinity_df = pd.read_csv(INPUT_DIR / "product_affinity_week2.csv")
if "description_a" in affinity_df.columns:
    affinity_df["description_a"] = affinity_df["description_a"].map(clean_product_name)
if "description_b" in affinity_df.columns:
    affinity_df["description_b"] = affinity_df["description_b"].map(clean_product_name)
affinity_df = affinity_df.rename(columns=AFFINITY_COL_MAP)

out_path = SUPABASE_DIR / CSV_FILE_RENAME_MAP["product_affinity_week2.csv"]
affinity_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved {out_path.name}: {affinity_df.shape}")
print("Headings:", list(affinity_df.columns))
display(affinity_df.head(5))

Saved product_bundles.csv: (51240, 11)
Headings: ['Product A Code', 'Product A', 'Product B Code', 'Product B', 'Orders Together', 'Product A Orders', 'Product B Orders', 'Order Share (%)', 'Buy A -> Buy B (%)', 'Buy B -> Buy A (%)', 'Bundle Lift']


,Product A Code,Product A,Product B Code,Product B,Orders Together,Product A Orders,Product B Orders,Order Share (%),Buy A -> Buy B (%),Buy B -> Buy A (%),Bundle Lift
0,21733,Red Hanging Heart T-Light Holder,85123A,White Hanging Heart T-Light Holder,722,994,3021,3.8068,72.64,23.90,4.56
1,21231,Sweetheart Ceramic Trinket Box,21232,Strawberry Ceramic Trinket Box,622,808,1335,3.2796,76.98,46.59,10.94
2,21212,Pack of 72 Retrospot Cake Cases,84991,60 Teatime Fairy Cake Cases,601,1522,1136,3.1688,39.49,52.90,6.59
3,82482,Wooden Picture Frame White Finish,82494L,Wooden Frame Antique White,554,818,1009,2.9210,67.73,54.91,12.73
4,85099B,Red Retrospot Jumbo Bag,85099F,Jumbo Bag Strawberry,549,1708,876,2.8947,32.14,62.67,6.96


In [7]:
# Process 5/5: cohort_retention_week2.csv -> cohort_retention.csv
# Keep Months Since First Purchase numeric (0..12) so SQL sorting and cohort heatmaps remain ordered.
cohort_df = pd.read_csv(INPUT_DIR / "cohort_retention_week2.csv")
cohort_df = cohort_df.rename(columns=COHORT_COL_MAP)

out_path = SUPABASE_DIR / CSV_FILE_RENAME_MAP["cohort_retention_week2.csv"]
cohort_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved {out_path.name}: {cohort_df.shape}")
print("Headings:", list(cohort_df.columns))
display(cohort_df.head(5))

Saved cohort_retention.csv: (91, 5)
Headings: ['First Purchase Month', 'Months Since First Purchase', 'Active Customers', 'Cohort Size', 'Retention Rate (%)']


,First Purchase Month,Months Since First Purchase,Active Customers,Cohort Size,Retention Rate (%)
0,2009-12,0,951,951,100.00
1,2009-12,1,333,951,35.02
2,2009-12,2,317,951,33.33
3,2009-12,3,404,951,42.48
4,2009-12,4,360,951,37.85


In [8]:
# Final verification of all Supabase-ready CSVs in data/supabase_db/
summary = []
for src_name, dst_name in CSV_FILE_RENAME_MAP.items():
    fpath = SUPABASE_DIR / dst_name
    df_check = pd.read_csv(fpath)
    summary.append({
        "original_file": src_name,
        "supabase_file": str(fpath),
        "rows": len(df_check),
        "columns": len(df_check.columns),
        "sample_headings": ", ".join(df_check.columns[:4]) + " ...",
        "size_kb": round(fpath.stat().st_size / 1024, 1),
    })

display(pd.DataFrame(summary))

,original_file,supabase_file,rows,columns,sample_headings,size_kb
0,cleaned_retail_week1.csv,c:\Users\acer\Downloads\Consumer-360\data\supa...,399568,11,"Invoice No, Product Code, Product Name, Quanti...",42161.2
1,customer_rfm_week1.csv,c:\Users\acer\Downloads\Consumer-360\data\supa...,4285,12,"Customer ID, Last Purchase, Total Orders, Tota...",312.1
2,customer_segments_week2.csv,c:\Users\acer\Downloads\Consumer-360\data\supa...,4285,14,"Customer ID, Country, Last Purchase, Days Inac...",468.8
3,product_affinity_week2.csv,c:\Users\acer\Downloads\Consumer-360\data\supa...,51240,11,"Product A Code, Product A, Product B Code, Pro...",5080.1
4,cohort_retention_week2.csv,c:\Users\acer\Downloads\Consumer-360\data\supa...,91,5,"First Purchase Month, Months Since First Purch...",2.3
